# StreamFlix Content Analytics
## Phase 1 — Data Cleaning & Quality Check

**Analyst:** Kajal Rajak

**Project Overview:** This project analyses StreamFlix subscriber profiles, content catalogue, viewing sessions, ratings, reviews, and watchlists.  
Phase 1 profiles the raw data, applies safe and traceable cleaning, validates data quality and relationships, and exports cleaned CSV files for EDA and KPI analysis.

## Step 1 — Import Required Libraries

In [27]:
import os
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

## Step 2 — Extract the Dataset

In [28]:
ZIP_PATH = Path("ff5a102e4361911d42cef9e21cda23d0.zip")
EXTRACT_DIR = Path("streamflix_raw_data")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted to:", EXTRACT_DIR.resolve())
print("Extracted files:", sorted(p.name for p in EXTRACT_DIR.glob("*")))

Dataset extracted to: C:\Users\91916\OneDrive\문서\Jupyter_Notebook\Streamflix-Content-Analytics-Project\streamflix_raw_data
Extracted files: ['StreamFlix_Data_Dictionary.docx', 'StreamFlix_ERD.png', 'StreamFlix_Project_PRD (2).docx', 'ratings.csv', 'reviews.csv', 'schema_and_sql (2).sql', 'subscribers.csv', 'titles.csv', 'watch_history.csv', 'watchlist.csv']


## Step 3 — Load All Six CSV Files

In [29]:
csv_files = {
    "subscribers": "subscribers.csv",
    "titles": "titles.csv",
    "watch_history": "watch_history.csv",
    "ratings": "ratings.csv",
    "reviews": "reviews.csv",
    "watchlist": "watchlist.csv",
}

raw_data_dict = {}
for table_name, file_name in csv_files.items():
    file_path = EXTRACT_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"Missing expected file: {file_path}")
    raw_data_dict[table_name] = pd.read_csv(file_path, low_memory=False)

for table_name, df in raw_data_dict.items():
    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

subscribers: 15,000 rows × 14 columns
titles: 9,000 rows × 21 columns
watch_history: 650,000 rows × 10 columns
ratings: 130,000 rows × 5 columns
reviews: 110,000 rows × 7 columns
watchlist: 65,000 rows × 5 columns


## Step 4 — Initial Data Profile

In [30]:
profile_rows = []
for table_name, df in raw_data_dict.items():
    profile_rows.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "total_missing_cells": int(df.isna().sum().sum()),
    })

initial_profile = pd.DataFrame(profile_rows)
display(initial_profile)

# Column-level data types and missing-value profile
for table_name, df in raw_data_dict.items():
    print(f"\\n--- {table_name} ---")
    display(pd.DataFrame({
        "column": df.columns,
        "data_type": [str(df[col].dtype) for col in df.columns],
        "missing_count": [int(df[col].isna().sum()) for col in df.columns],
        "missing_pct": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    }))

,table,rows,columns,exact_duplicate_rows,total_missing_cells
0,subscribers,15000,14,0,11199
1,titles,9000,21,0,1966
2,watch_history,650000,10,0,0
3,ratings,130000,5,0,0
4,reviews,110000,7,0,0
5,watchlist,65000,5,0,0


\n--- subscribers ---


,column,data_type,missing_count,missing_pct
0,subscriber_id,object,0,0.00
1,signup_date,object,0,0.00
2,country,object,0,0.00
3,region,object,0,0.00
4,age,int64,0,0.00
5,gender,object,0,0.00
6,plan_type,object,0,0.00
7,monthly_price_usd,float64,0,0.00
8,household_size,int64,0,0.00
9,primary_device,object,0,0.00


\n--- titles ---


,column,data_type,missing_count,missing_pct
0,title_id,object,0,0.00
1,title_name,object,0,0.00
2,type,object,0,0.00
3,primary_genre,object,0,0.00
4,country,object,0,0.00
5,language,object,0,0.00
6,release_year,int64,0,0.00
7,date_added,object,0,0.00
8,maturity_rating,object,0,0.00
9,seasons,int64,0,0.00


\n--- watch_history ---


... (3 outputs left out)

\n--- reviews ---


,column,data_type,missing_count,missing_pct
0,review_id,int64,0,0.0
1,subscriber_id,object,0,0.0
2,title_id,object,0,0.0
3,review_text,object,0,0.0
4,sentiment,object,0,0.0
5,helpful_votes,int64,0,0.0
6,review_date,object,0,0.0


\n--- watchlist ---


,column,data_type,missing_count,missing_pct
0,watchlist_id,int64,0,0.0
1,subscriber_id,object,0,0.0
2,title_id,object,0,0.0
3,added_date,object,0,0.0
4,watched,bool,0,0.0


**Observation:** The dataset contains six tables: `subscribers` (15,000 rows), `titles` (9,000 rows), `watch_history` (650,000 rows), `ratings` (130,000 rows), `reviews` (110,000 rows), and `watchlist` (65,000 rows). The initial profiling identified missing values in two columns: `subscribers.churn_date` (11,199 missing values, 74.66%) and `titles.license_expiry` (1,966 missing values, 21.84%). All other columns have no missing values. The data types and duplicate records will be examined further during the cleaning process.

## Step 5 — Create Working Copies and Standardise Text

In [31]:
cleaned_data_dict = {name: df.copy() for name, df in raw_data_dict.items()}

for table_name, df in cleaned_data_dict.items():
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].map(lambda value: value.strip() if isinstance(value, str) else value)

category_columns = {
    "subscribers": ["country", "region", "gender", "plan_type", "primary_device", "payment_method"],
    "titles": ["type", "primary_genre", "country", "language", "maturity_rating", "license_type"],
    "watch_history": ["device", "region"],
    "reviews": ["sentiment"],
}
for table_name, columns in category_columns.items():
    df = cleaned_data_dict[table_name]
    for col in columns:
        if col in df.columns:
            df[col] = df[col].map(
                lambda value: value.title() if isinstance(value, str) else value
            )

if "sentiment" in cleaned_data_dict["reviews"].columns:
    sentiment_map = {"Positive": "Positive", "Neutral": "Neutral", "Negative": "Negative"}
    cleaned_data_dict["reviews"]["sentiment"] = cleaned_data_dict["reviews"]["sentiment"].replace(sentiment_map)

print("Text trimming and selected category standardisation complete.")

Text trimming and selected category standardisation complete.


## Step 6 — Convert Date and Numeric Columns

In [32]:
date_columns = {
    "subscribers": ["signup_date", "churn_date"],
    "titles": ["date_added", "license_expiry"],
    "watch_history": ["watch_date"],
    "ratings": ["rating_date"],
    "reviews": ["review_date"],
    "watchlist": ["added_date"],
}

numeric_columns = {
    "subscribers": ["age", "monthly_price_usd", "household_size", "tenure_months"],
    "titles": [
        "release_year", "seasons", "content_duration_min", "quality_score",
        "popularity_score", "license_cost_usd", "total_watch_hours", "total_plays"
    ],
    "watch_history": [
        "watch_id", "content_duration_min", "watch_duration_min", "completion_pct"
    ],
    "ratings": ["rating_id", "rating"],
    "reviews": ["review_id", "helpful_votes"],
    "watchlist": ["watchlist_id"],
}

for table_name, columns in date_columns.items():
    df = cleaned_data_dict[table_name]
    for col in columns:
        if col in df.columns:
            # Invalid date strings become NaT and can be counted in the QC report.
            df[col] = pd.to_datetime(df[col], errors="coerce")

for table_name, columns in numeric_columns.items():
    df = cleaned_data_dict[table_name]
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

print("Date and numeric conversions complete.")
for table_name, columns in date_columns.items():
    print(table_name, {col: str(cleaned_data_dict[table_name][col].dtype) for col in columns})

Date and numeric conversions complete.
subscribers {'signup_date': 'datetime64[ns]', 'churn_date': 'datetime64[ns]'}
titles {'date_added': 'datetime64[ns]', 'license_expiry': 'datetime64[ns]'}
watch_history {'watch_date': 'datetime64[ns]'}
ratings {'rating_date': 'datetime64[ns]'}
reviews {'review_date': 'datetime64[ns]'}
watchlist {'added_date': 'datetime64[ns]'}


In [33]:
# Check for invalid dates after conversion
for table_name, columns in date_columns.items():
    for col in columns:
        invalid_dates = cleaned_data_dict[table_name][col].isna().sum()
        print(f"{table_name}.{col}: {invalid_dates} missing or invalid dates")

subscribers.signup_date: 0 missing or invalid dates
subscribers.churn_date: 11199 missing or invalid dates
titles.date_added: 0 missing or invalid dates
titles.license_expiry: 1966 missing or invalid dates
watch_history.watch_date: 0 missing or invalid dates
ratings.rating_date: 0 missing or invalid dates
reviews.review_date: 0 missing or invalid dates
watchlist.added_date: 0 missing or invalid dates


**Observation:** The missing-value analysis identified 11,199 missing values in `subscribers.churn_date`, all associated with active subscribers. These values are retained because active subscribers may not have a churn date. Additionally, 1,966 missing values were identified in `titles.license_expiry`, all associated with titles classified as `Original`. These values are retained pending confirmation of the licensing rules in the PRD.


## Step 7 — Check and Remove Exact Duplicate Rows

In [34]:
duplicate_log = []
for table_name, df in cleaned_data_dict.items():
    duplicate_count = int(df.duplicated().sum())
    duplicate_log.append({
        "table": table_name,
        "exact_duplicate_rows_removed": duplicate_count
    })
    if duplicate_count:
        cleaned_data_dict[table_name] = df.drop_duplicates().copy()

display(pd.DataFrame(duplicate_log))

,table,exact_duplicate_rows_removed
0,subscribers,0
1,titles,0
2,watch_history,0
3,ratings,0
4,reviews,0
5,watchlist,0


**Observation:** No exact duplicate rows were identified in any of the six tables. Therefore, no rows were removed during exact duplicate removal. Primary-key duplicates and other data-quality issues will be checked separately.


## Step 8 — Check Duplicate Primary Keys

In [35]:
primary_keys = {
    "subscribers": "subscriber_id",
    "titles": "title_id",
    "watch_history": "watch_id",
    "ratings": "rating_id",
    "reviews": "review_id",
    "watchlist": "watchlist_id",
}

primary_key_log = []
for table_name, key_col in primary_keys.items():
    df = cleaned_data_dict[table_name]
    duplicate_key_mask = df[key_col].duplicated(keep="first") & df[key_col].notna()
    duplicate_key_count = int(duplicate_key_mask.sum())
    primary_key_log.append({
        "table": table_name,
        "primary_key": key_col,
        "duplicate_key_rows": duplicate_key_count
    })
    if duplicate_key_count:
        cleaned_data_dict[table_name] = df.loc[~duplicate_key_mask].copy()

display(pd.DataFrame(primary_key_log))

,table,primary_key,duplicate_key_rows
0,subscribers,subscriber_id,0
1,titles,title_id,0
2,watch_history,watch_id,0
3,ratings,rating_id,0
4,reviews,review_id,0
5,watchlist,watchlist_id,0


## Step 9 — Subscriber Churn Date Validation

In [36]:
qc_findings = []

def add_qc(table, check, issue_count, details):
    qc_findings.append({
        "table": table,
        "check": check,
        "issue_count": int(issue_count),
        "details": details
    })

In [37]:
# Check whether inactive subscribers have a churn date after signup date.

subscribers = cleaned_data_dict["subscribers"]

bad_churn_order = (
    subscribers["is_active"].eq(False)
    & subscribers["churn_date"].notna()
    & subscribers["signup_date"].notna()
    & (subscribers["churn_date"] <= subscribers["signup_date"])
)

add_qc(
    "subscribers",
    "Inactive subscriber churn_date must be after signup_date",
    bad_churn_order.sum(),
    "Rows flagged where churn date is on/before signup date."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
0,subscribers,Inactive subscriber churn_date must be after s...,0,Rows flagged where churn date is on/before sig...


## Step 10 — Subscriber Churn Date Consistency

In [38]:
# Check active and inactive subscribers for missing or unexpected churn dates.

active_with_churn_date = (
    subscribers["is_active"].eq(True)
    & subscribers["churn_date"].notna()
)

inactive_without_churn_date = (
    subscribers["is_active"].eq(False)
    & subscribers["churn_date"].isna()
)

add_qc(
    "subscribers",
    "Active subscriber has churn_date",
    active_with_churn_date.sum(),
    "Active rows with a populated churn date."
)

add_qc(
    "subscribers",
    "Inactive subscriber has no churn_date",
    inactive_without_churn_date.sum(),
    "Inactive rows with a blank/unparseable churn date."
)

display(pd.DataFrame(qc_findings).tail(2))

,table,check,issue_count,details
1,subscribers,Active subscriber has churn_date,0,Active rows with a populated churn date.
2,subscribers,Inactive subscriber has no churn_date,0,Inactive rows with a blank/unparseable churn d...


## Step 11 — Short Tenure Validation

In [39]:
# Informational check for active subscribers with tenure of one month or less.

short_active_tenure = (
    subscribers["is_active"].eq(True)
    & subscribers["tenure_months"].le(1)
)

add_qc(
    "subscribers",
    "Active subscribers with tenure_months <= 1",
    short_active_tenure.sum(),
    "Informational only; short tenure can be valid."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
3,subscribers,Active subscribers with tenure_months <= 1,321,Informational only; short tenure can be valid.


## Step 12 — Watch Duration Validation

In [40]:
# Identify viewing sessions longer than the content duration.

watch_history = cleaned_data_dict["watch_history"]

duration_exceeds_content = (
    watch_history["content_duration_min"].gt(0)
    & watch_history["watch_duration_min"].gt(
        watch_history["content_duration_min"]
    )
)

add_qc(
    "watch_history",
    "watch_duration_min exceeds content_duration_min",
    duration_exceeds_content.sum(),
    "Flagged; sessions can exceed content length due to data issues or repeat viewing."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
4,watch_history,watch_duration_min exceeds content_duration_min,0,Flagged; sessions can exceed content length du...


## Step 13 — Completion Percentage Validation

In [41]:
# Compare recorded completion percentage with duration-based completion.

expected_completion = (
    watch_history["watch_duration_min"]
    / watch_history["content_duration_min"].replace(0, np.nan)
    * 100
)

completion_mismatch = (
    expected_completion.notna()
    & watch_history["completion_pct"].notna()
    & (
        (watch_history["completion_pct"] - expected_completion).abs() > 2.0
    )
)

add_qc(
    "watch_history",
    "completion_pct differs from duration-based percentage by > 2 points",
    completion_mismatch.sum(),
    "Flagged using a 2-percentage-point tolerance; original values are retained."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
5,watch_history,completion_pct differs from duration-based per...,0,Flagged using a 2-percentage-point tolerance; ...


## Step 14 — Completion Percentage Range Check

In [42]:
# Check whether completion percentages fall within the 0–100 range.

invalid_completion = (
    watch_history["completion_pct"].notna()
    & ~watch_history["completion_pct"].between(0, 100)
)

add_qc(
    "watch_history",
    "completion_pct outside 0–100",
    invalid_completion.sum(),
    "Values outside the documented range."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
6,watch_history,completion_pct outside 0–100,0,Values outside the documented range.


## Step 15 — Rating Validation

In [43]:
# Check whether ratings fall within the allowed 1–5 range.

ratings = cleaned_data_dict["ratings"]

invalid_rating = (
    ratings["rating"].notna()
    & ~ratings["rating"].between(1, 5)
)

add_qc(
    "ratings",
    "rating outside 1–5",
    invalid_rating.sum(),
    "Invalid ratings flagged."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
7,ratings,rating outside 1–5,0,Invalid ratings flagged.


## Step 16 — Review Sentiment Validation

In [44]:
# Check whether sentiment values match the allowed categories.

reviews = cleaned_data_dict["reviews"]

allowed_sentiments = {"Positive", "Neutral", "Negative"}

invalid_sentiment = (
    reviews["sentiment"].notna()
    & ~reviews["sentiment"].isin(allowed_sentiments)
)

add_qc(
    "reviews",
    "sentiment outside Positive/Neutral/Negative",
    invalid_sentiment.sum(),
    "Invalid sentiment labels flagged; not guessed or replaced."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
8,reviews,sentiment outside Positive/Neutral/Negative,0,Invalid sentiment labels flagged; not guessed ...


## Step 17 — Subscriber and Title ID Reference Validation

In [45]:
# Identify subscriber_id and title_id values that do not exist in their parent tables.

titles = cleaned_data_dict["titles"]

subscriber_ids = set(subscribers["subscriber_id"].dropna())
title_ids = set(titles["title_id"].dropna())

for table_name, df in {
    "watch_history": watch_history,
    "ratings": ratings,
    "reviews": reviews,
    "watchlist": cleaned_data_dict["watchlist"],
}.items():

    bad_subscriber_fk = (
        df["subscriber_id"].notna()
        & ~df["subscriber_id"].isin(subscriber_ids)
    )

    bad_title_fk = (
        df["title_id"].notna()
        & ~df["title_id"].isin(title_ids)
    )

    add_qc(
        table_name,
        "subscriber_id missing from subscribers",
        bad_subscriber_fk.sum(),
        "Foreign-key rows flagged."
    )

    add_qc(
        table_name,
        "title_id missing from titles",
        bad_title_fk.sum(),
        "Foreign-key rows flagged."
    )

display(pd.DataFrame(qc_findings).tail(8))

,table,check,issue_count,details
9,watch_history,subscriber_id missing from subscribers,0,Foreign-key rows flagged.
10,watch_history,title_id missing from titles,0,Foreign-key rows flagged.
11,ratings,subscriber_id missing from subscribers,0,Foreign-key rows flagged.
12,ratings,title_id missing from titles,0,Foreign-key rows flagged.
13,reviews,subscriber_id missing from subscribers,0,Foreign-key rows flagged.
14,reviews,title_id missing from titles,0,Foreign-key rows flagged.
15,watchlist,subscriber_id missing from subscribers,0,Foreign-key rows flagged.
16,watchlist,title_id missing from titles,0,Foreign-key rows flagged.


## Step 18 — Watchlist Reference Validation

In [46]:
# Check whether each watchlist entry references an existing subscriber and title.

watchlist = cleaned_data_dict["watchlist"]

watchlist_bad_fk = (
    (
        watchlist["subscriber_id"].notna()
        & ~watchlist["subscriber_id"].isin(subscriber_ids)
    )
    |
    (
        watchlist["title_id"].notna()
        & ~watchlist["title_id"].isin(title_ids)
    )
)

add_qc(
    "watchlist",
    "Watchlist entry has invalid subscriber or title link",
    watchlist_bad_fk.sum(),
    "Rows flagged if either foreign key does not exist."
)

display(pd.DataFrame(qc_findings).tail(1))

,table,check,issue_count,details
17,watchlist,Watchlist entry has invalid subscriber or titl...,0,Rows flagged if either foreign key does not ex...


## Step 19 — Generate the Final Data Quality Report

In [47]:
# Combining all data quality findings into a single report.
qc_report = pd.DataFrame(qc_findings)

display(qc_report)

,table,check,issue_count,details
0,subscribers,Inactive subscriber churn_date must be after s...,0,Rows flagged where churn date is on/before sig...
1,subscribers,Active subscriber has churn_date,0,Active rows with a populated churn date.
2,subscribers,Inactive subscriber has no churn_date,0,Inactive rows with a blank/unparseable churn d...
3,subscribers,Active subscribers with tenure_months <= 1,321,Informational only; short tenure can be valid.
4,watch_history,watch_duration_min exceeds content_duration_min,0,Flagged; sessions can exceed content length du...
5,watch_history,completion_pct differs from duration-based per...,0,Flagged using a 2-percentage-point tolerance; ...
6,watch_history,completion_pct outside 0–100,0,Values outside the documented range.
7,ratings,rating outside 1–5,0,Invalid ratings flagged.
8,reviews,sentiment outside Positive/Neutral/Negative,0,Invalid sentiment labels flagged; not guessed ...
9,watch_history,subscriber_id missing from subscribers,0,Foreign-key rows flagged.


## Step 20 — Build a Column-Level Missing-Value and Final Profile Report

In [48]:
missing_report_rows = []
final_profile_rows = []

for table_name, df in cleaned_data_dict.items():
    final_profile_rows.append({
        "table": table_name,
        "rows_after_cleaning": len(df),
        "columns": len(df.columns),
        "exact_duplicate_rows_remaining": int(df.duplicated().sum()),
    })
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        missing_report_rows.append({
            "table": table_name,
            "column": col,
            "missing_count": missing_count,
            "missing_pct": round(missing_count / len(df) * 100, 2) if len(df) else 0,
            "dtype_after_cleaning": str(df[col].dtype),
        })

final_profile = pd.DataFrame(final_profile_rows)
missing_report = pd.DataFrame(missing_report_rows)

display(final_profile)
display(missing_report[missing_report["missing_count"] > 0].sort_values(
    ["table", "missing_pct"], ascending=[True, False]
))

,table,rows_after_cleaning,columns,exact_duplicate_rows_remaining
0,subscribers,15000,14,0
1,titles,9000,21,0
2,watch_history,650000,10,0
3,ratings,130000,5,0
4,reviews,110000,7,0
5,watchlist,65000,5,0


,table,column,missing_count,missing_pct,dtype_after_cleaning
13,subscribers,churn_date,11199,74.66,datetime64[ns]
32,titles,license_expiry,1966,21.84,datetime64[ns]


## Step 21 — Save Cleaned CSV Files and Quality Reports

In [49]:
CLEANED_DIR = Path("streamflix_cleaned_data")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

for table_name, df in cleaned_data_dict.items():
    df.to_csv(CLEANED_DIR / f"{table_name}_cleaned.csv", index=False)

qc_report.to_csv(CLEANED_DIR / "data_quality_checks.csv", index=False)
missing_report.to_csv(CLEANED_DIR / "missing_value_report.csv", index=False)
final_profile.to_csv(CLEANED_DIR / "final_data_profile.csv", index=False)
pd.DataFrame(duplicate_log).to_csv(CLEANED_DIR / "exact_duplicate_log.csv", index=False)
pd.DataFrame(primary_key_log).to_csv(CLEANED_DIR / "primary_key_duplicate_log.csv", index=False)

print("Cleaned data and QC reports saved in:", CLEANED_DIR.resolve())
print("Files:", sorted(p.name for p in CLEANED_DIR.glob("*.csv")))

Cleaned data and QC reports saved in: C:\Users\91916\OneDrive\문서\Jupyter_Notebook\Streamflix-Content-Analytics-Project\streamflix_cleaned_data
Files: ['data_quality_checks.csv', 'exact_duplicate_log.csv', 'final_data_profile.csv', 'missing_value_report.csv', 'primary_key_duplicate_log.csv', 'ratings_cleaned.csv', 'reviews_cleaned.csv', 'subscribers_cleaned.csv', 'titles_cleaned.csv', 'watch_history_cleaned.csv', 'watchlist_cleaned.csv']


## Step 12 — Export Cleaned Tables to PostgreSQL

In [50]:
from sqlalchemy import create_engine
from getpass import getpass
from urllib.parse import quote_plus

DB_USER = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "streamflix_analytics"
DB_PASSWORD = getpass("Enter PostgreSQL password: ")

connection_url = (
    f"postgresql+psycopg2://{quote_plus(DB_USER)}:{quote_plus(DB_PASSWORD)}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
engine = create_engine(connection_url)

for table_name, df in cleaned_data_dict.items():
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=10_000,
        method="multi"
    )
    print(f"Loaded {table_name}: {len(df):,} rows")

engine.dispose()
print("PostgreSQL export complete.")

Enter PostgreSQL password:  ········


Loaded subscribers: 15,000 rows
Loaded titles: 9,000 rows
Loaded watch_history: 650,000 rows
Loaded ratings: 130,000 rows
Loaded reviews: 110,000 rows
Loaded watchlist: 65,000 rows
PostgreSQL export complete.


## Step 13 — Phase 1 Summary

Review the outputs generated during Phase 1 to summarize the data-cleaning process and final data quality. The summary should cover:

* **Dataset overview:** Compare row and column counts before and after cleaning for all six tables.
* **Duplicate records:** Report the exact duplicate rows and repeated primary-key records identified and removed, if any.
* **Missing values:** Document missing values by table and column, including their counts and percentages.
* **Subscriber validation:** Verify churn-date consistency, including the relationship between signup dates, churn dates, and subscriber status.
* **Viewing activity validation:** Check viewing durations against content durations and validate completion percentages against the expected range and duration-based calculations.
* **Ratings and reviews:** Identify ratings outside the permitted 1–5 range and review sentiments outside the allowed categories.
* **Referential integrity:** Report invalid subscriber and title references in `watch_history`, `ratings`, `reviews`, and `watchlist`.

### Important Notes

Data-quality checks flag potential issues for review rather than automatically modifying business values. Review `data_quality_checks.csv` and other generated quality reports before deciding whether any flagged records require correction or exclusion.

For Phase 2 (Exploratory Data Analysis) and Phase 3 (Dashboard Development), use the validated data in `cleaned_data_dict` or the exported `*_cleaned.csv` files. Retain legitimate missing values where appropriate, and handle them according to the relevant business rules when calculating KPIs and preparing visualizations.